In [ ]:
import numpy as np
import scipy.integrate as integrate
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import os
import pandas as pd
import time


from scipy.interpolate import griddata

In [ ]:
print(torch.__version__)

In [ ]:

# Use CUDA when it is available.
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    
print(device)

In [ ]:
torch.cuda.set_device(1)
torch.manual_seed(1234)
np.random.seed(1234)

In [ ]:
start_time = time.time()

In [ ]:
checkpoint_dir = os.path.dirname(r"/home/u/reynaquita2905/RD2/SR/weights_adam/example1_oscillatory/")
checkpoint_path = os.path.join(checkpoint_dir, "run_3")

checkpoint_path_ffm = os.path.join(checkpoint_path, "ffm", "best-cp.ckpt")
checkpoint_path_ffp = os.path.join(checkpoint_path, "ffp", "best-cp.ckpt")


In [ ]:
number_layers_1 = 10
number_layers_2 = 8
number_neurons = 40
output_size = 1

In [ ]:
def init_model(input_size, layers, output_size = output_size, layer_width = number_neurons):
    model = nn.Sequential() 
    model.add_module("dense_0", nn.Linear(input_size, layer_width))
    model.add_module("activation_0", nn.Tanh())
    
    for i in range(1, layers):
        model.add_module('dense_%d' % i, nn.Linear(layer_width, layer_width))
        model.add_module('activation_%d' % i, nn.Tanh())

    model.add_module('dense_' + str(i+1), nn.Linear(layer_width, output_size))
    model.add_module('activation_%d' % i, nn.Sigmoid())
    
    return model

In [ ]:
ffm = init_model(input_size = 2, layers = number_layers_1)
ffm = ffm.to(device)
ffm.load_state_dict(torch.load(checkpoint_path_ffm))
ffm.eval()

ffp = init_model(input_size = 2, layers = number_layers_2 )
ffp = ffp.to(device)
ffp.load_state_dict(torch.load(checkpoint_path_ffp))
ffp.eval()

In [ ]:
delta = 5e-2

a_t = 0
b_t = 0.5
a_xi = - delta
b_xi = delta


number_of_grid_xi = 5000
number_of_grid_t = 1000

N  = (b_xi - a_xi)*number_of_grid_xi
Nt = (b_t - a_t)*number_of_grid_t
h_ = (b_xi - a_xi)/N
k = (b_t - a_t)/Nt

t = np.arange(a_t, b_t, k)
xi = np.arange(a_xi, b_xi, h_)
Xi, T = np.meshgrid(xi, t)

In [ ]:
C_star = 0.1
eta = 1

# Define the interface position and velocity.
h = lambda t, c = C_star, eta = eta, b = -17 : eta * t + 0.1 * np.sin(b*t) + c
h_prime = lambda t, eta = eta, b = -17 : np.ones(t.shape) * eta + 0.1 * b * np.cos(b * t)




In [ ]:
X_star_ = np.hstack((Xi.flatten()[:,None], T.flatten()[:,None]))

cond = X_star_[:,0:1] < 0
X_star_1 = np.where(cond, X_star_, np.nan)
X_star_1 = X_star_1[~np.isnan(X_star_1).any(axis=1)]
X_star_1_ = torch.tensor(X_star_1, requires_grad=False).float().to(device)

X_star_2 = np.where(cond, np.nan, X_star_)
X_star_2 = X_star_2[~np.isnan(X_star_2).any(axis=1)]
X_star_2_ = torch.tensor(X_star_2, requires_grad=False).float().to(device)

In [ ]:
Um = ffm(X_star_1_)
Up = ffp(X_star_2_)

Um = Um.cpu().detach().numpy()
Up = Up.cpu().detach().numpy()

X_star = np.concatenate([X_star_1, X_star_2])
u_pred = np.concatenate([Um, Up])

In [ ]:
# Interpolate predictions onto the evaluation grid.
U_pred = griddata(X_star, u_pred.flatten(), (Xi, T), method='cubic')


In [ ]:
U_exact = np.load(checkpoint_path + r"/U_exact.npy")

print(U_exact.shape)

In [ ]:
# Align the reference solution with the evaluation time grid.
gap = int(U_exact.shape[0] / len(t))
print(gap)

indices = np.arange(0,U_exact.shape[0],gap)


U_exact_new = U_exact[indices]
print(U_exact_new.shape)

In [ ]:
# Transform the evaluation grid to physical coordinates.
X = np.zeros((Xi.shape))

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        X[i,j] = Xi[i,j] + h(T[i,j])

In [ ]:

ts = [0.00, 0.0100, 0.025, 0.4]



In [ ]:
fig, axs = plt.subplots(2, 2,figsize=(10,8))


axs[0, 0].plot(xi,U_exact_new[int((ts[0]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')
axs[0, 0].plot(xi,U_pred[int((ts[0]/b_t)*(len(t)-1)),:], linewidth = 2, linestyle = '--', color = 'r', label = 'PINN')
axs[0, 0].set_title('$t = $' + str(ts[0]), fontsize = 10)

axs[0, 0].set_ylim([0.95  - 0.002, 1 + 0.002])


axs[0, 0].grid()


axs[0, 1].plot(xi,U_exact_new[int((ts[1]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')
axs[0, 1].plot(xi,U_pred[int((ts[1]/b_t)*(len(t)-1)),:], linewidth = 2, linestyle = '--', color = 'r', label = 'PINN')
axs[0, 1].set_title('$t = $' + str(ts[1]), fontsize = 10)

axs[0, 1].set_ylim([0.95  - 0.002, 1 + 0.002])


axs[0, 1].grid()


axs[1, 0].plot(xi,U_exact_new[int((ts[2]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')
axs[1, 0].plot(xi,U_pred[int((ts[2]/b_t)*(len(t)-1)),:], linewidth = 2, linestyle = '--', color = 'r', label = 'PINN')
axs[1, 0].set_title('$t = $'+ str(ts[2]), fontsize = 10)

axs[1, 0].set_ylim([0.95  - 0.002, 1 + 0.002])


axs[1, 0].grid()


axs[1, 1].plot(xi,U_exact_new[int((ts[3]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')
axs[1, 1].plot(xi,U_pred[int((ts[3]/b_t)*(len(t)-1)),:], linewidth = 2, linestyle = '--', color = 'r', label = 'PINN')
axs[1, 1].set_title('$t = $' + str(ts[3]), fontsize = 10)

axs[1, 1].set_ylim([0.95  - 0.002, 1 + 0.002])


axs[1, 1].grid()

for ax in axs.flat:
    ax.set(xlabel=r'$\xi$', ylabel=r'$U(\xi,t)$')
    
plt.legend()
fig.tight_layout()
plt.savefig(checkpoint_path + "/uplot_xi.pdf", format="pdf", bbox_inches="tight")
plt.savefig(checkpoint_path + "/uplot_xi.png", bbox_inches="tight")
plt.show() 

In [ ]:
error = np.absolute(U_exact_new[:-1,:] - U_pred[:-1,:])

rel_l2_norm = np.linalg.norm(error,2)/np.linalg.norm(U_exact_new[:-1,:],2)
print("Relative L2 norm: ", rel_l2_norm)

l2_norm = np.linalg.norm(error, 2)
print("L2 norm: ", l2_norm)

In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print(elapsed_time)

with open(checkpoint_path + "/time_test_cpinn.txt", 'w') as file:
    file.write(f"Time: {str(elapsed_time)}") 

In [ ]:

# Compare solutions and absolute error in physical coordinates.
n_t = min(U_pred.shape[0], U_exact_new.shape[0], Xi.shape[0])
n_x = min(U_pred.shape[1], U_exact_new.shape[1], Xi.shape[1])
X_plot = X[:n_t, :n_x]
T_plot = T[:n_t, :n_x]
U_pred_plot = U_pred[:n_t, :n_x]
U_exact_plot = U_exact_new[:n_t, :n_x]
abs_error_plot = np.abs(U_pred_plot - U_exact_plot)

solution_min = np.nanmin([U_pred_plot, U_exact_plot])
solution_max = np.nanmax([U_pred_plot, U_exact_plot])
if solution_min == solution_max:
    solution_min -= 1e-12
    solution_max += 1e-12
solution_levels = np.linspace(solution_min, solution_max, 41)
error_max = np.nanmax(abs_error_plot)
error_levels = np.linspace(0, error_max if error_max > 0 else 1e-12, 41)


interface_t = T_plot[:, 0]
interface_x = h(interface_t)
x_limits = (np.nanmin(X_plot), np.nanmax(X_plot))

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for ax, field, title in zip(axes[:2], [U_pred_plot, U_exact_plot], ['cPINN solution', 'Approximated solution']):
    contour = ax.contourf(X_plot, T_plot, field, levels=solution_levels, cmap='viridis', extend='both')
    ax.contour(X_plot, T_plot, field, levels=solution_levels[::5], colors='white', linewidths=0.35, alpha=0.6)
    ax.plot(interface_x, interface_t, color='cyan', linewidth=3, label='oscillatory interface:\n' + r'$x=t+0.1\sin(-17t)+0.1$')
    ax.set(title=title, xlabel='$x$', ylabel='$t$', xlim=x_limits, ylim=(a_t, b_t))
    ax.set_aspect('equal', adjustable='box')
    fig.colorbar(contour, ax=ax, label=r'$u(x,t)$')

error_contour = axes[2].contourf(X_plot, T_plot, abs_error_plot, levels=error_levels, cmap='magma', extend='max')
axes[2].contour(X_plot, T_plot, abs_error_plot, levels=error_levels[::5], colors='white', linewidths=0.35, alpha=0.6)
axes[2].plot(interface_x, interface_t, color='cyan', linewidth=3, label='oscillatory interface:\n' + r'$x=t+0.1\sin(-17t)+0.1$')
axes[2].set(title='Absolute error', xlabel='$x$', ylabel='$t$', xlim=x_limits, ylim=(a_t, b_t))
axes[2].set_aspect('equal', adjustable='box')
fig.colorbar(error_contour, ax=axes[2], label=r'$|u_{cPINN} - u_{Approximated}|$')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.06), frameon=True, ncol=1)
fig.savefig(checkpoint_path + '/solution_contours.pdf', format='pdf', bbox_inches='tight')
fig.savefig(checkpoint_path + '/solution_contours.png', bbox_inches='tight')
plt.show()


In [ ]:

n_t = min(U_pred.shape[0], U_exact_new.shape[0], Xi.shape[0])
n_x = min(U_pred.shape[1], U_exact_new.shape[1], Xi.shape[1])
# Compare solutions and absolute error in transformed coordinates.
Xi_plot = Xi[:n_t, :n_x]
T_plot = T[:n_t, :n_x]
U_pred_plot = U_pred[:n_t, :n_x]
U_exact_plot = U_exact_new[:n_t, :n_x]
abs_error_plot = np.abs(U_pred_plot - U_exact_plot)

solution_min = np.nanmin([U_pred_plot, U_exact_plot])
solution_max = np.nanmax([U_pred_plot, U_exact_plot])
if solution_min == solution_max:
    solution_min -= 1e-12
    solution_max += 1e-12
solution_levels = np.linspace(solution_min, solution_max, 41)
error_max = np.nanmax(abs_error_plot)
error_levels = np.linspace(0, error_max if error_max > 0 else 1e-12, 41)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
for ax, field, title in zip(axes[:2], [U_pred_plot, U_exact_plot], ['cPINN solution', 'Approximated solution']):
    contour = ax.contourf(Xi_plot, T_plot, field, levels=solution_levels, cmap='viridis', extend='both')
    ax.contour(Xi_plot, T_plot, field, levels=solution_levels[::5], colors='white', linewidths=0.35, alpha=0.6)
    ax.axvline(0, color='white', linestyle='--', linewidth=1.2, label='interface')
    ax.set(title=title, xlabel=r'$\xi$', ylabel='$t$', xlim=(a_xi, b_xi), ylim=(a_t, b_t))
    fig.colorbar(contour, ax=ax, label=r'$U(\xi,t)$')

error_contour = axes[2].contourf(Xi_plot, T_plot, abs_error_plot, levels=error_levels, cmap='magma', extend='max')
axes[2].contour(Xi_plot, T_plot, abs_error_plot, levels=error_levels[::5], colors='white', linewidths=0.35, alpha=0.6)
axes[2].axvline(0, color='cyan', linestyle='--', linewidth=1.2, label='interface')
axes[2].set(title='Absolute error', xlabel=r'$\xi$', ylabel='$t$', xlim=(a_xi, b_xi), ylim=(a_t, b_t))
fig.colorbar(error_contour, ax=axes[2], label=r'$|U_{cPINN} - U_{Approximated}|$')
fig.savefig(checkpoint_path + '/solution_contours1.pdf', format='pdf', bbox_inches='tight')
fig.savefig(checkpoint_path + '/solution_contours1.png', bbox_inches='tight')
plt.show()


In [ ]:
data1 = pd.read_csv(checkpoint_path + "/loss_adam.csv")
loss_m_list1 = data1['lossm'].tolist()
loss_p_list1 = data1['lossp'].tolist()

In [ ]:
loss_m_list = loss_m_list1
loss_p_list = loss_p_list1


In [ ]:
plt.figure(figsize=(10,6))
plt.plot(loss_m_list, linestyle = "--",color='blue', label = "loss 1" )
plt.plot(loss_p_list, linestyle = "--",color='red', label = "loss 2" )
plt.yscale('log', base=10)
plt.ylabel('loss') 
plt.xlabel('epochs')
plt.legend()
plt.grid()
plt.savefig(checkpoint_path + "/lossplot1.pdf", format="pdf", bbox_inches="tight")
plt.savefig(checkpoint_path + "/lossplot1.png", bbox_inches="tight")
plt.show() 

In [ ]:
with open (checkpoint_path + "/info2.txt", 'w') as file:  
    file.write("relative L2 norm: %s \n number of layers 1: %s \n number of layers 2: %s \n number of neurons: %s \n" % (str(rel_l2_norm), str(number_layers_1), str(number_layers_2), str(number_neurons)))

In [ ]:
np.save(checkpoint_path + r"/U_pred.npy", U_pred)
